In [1]:
%pip install -U pip
%pip install selenium webdriver-manager requests

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 3.7 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.8 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 3.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1
Note: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0/4 [python-dotenv]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   -------------------- ------------------- 2/4 [requests]
   ------------------------------ --------- 3/4 [webdriver-manager]
   ------------------------------ --------- 3/4 [webdriver-manager]
   ---------------------------------------- 4/4 [webdriver-manager]

Note: you may need to resta

In [2]:
import os
import json
import requests

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

In [ ]:
# Configuration (simple)
API_URL = os.getenv("API_URL", "http://localhost:5139/api")
FRONT_URL = os.getenv("FRONT_URL", "http://localhost:7004")

EMAIL = os.getenv("TEST_EMAIL", "admin@test.com")
PASSWORD = os.getenv("TEST_PASSWORD", "Admin@123")

# Inscription: on génère un email unique pour éviter les doublons
RUN_ID = os.getenv("RUN_ID") or __import__("datetime").datetime.now().strftime("%Y%m%d%H%M%S")
NEW_EMAIL = os.getenv("NEW_EMAIL", f"patient{RUN_ID}@test.com")
NEW_PASSWORD = os.getenv("NEW_PASSWORD", "Patient@123")

# Selenium
HEADLESS = os.getenv("HEADLESS", "0") == "1"

print("API_URL:", API_URL)
print("FRONT_URL:", FRONT_URL)
print("HEADLESS:", HEADLESS)
print("NEW_EMAIL:", NEW_EMAIL)

In [ ]:
def api_register(email: str, password: str, nom: str = "Test", prenom: str = "Patient", telephone: str = "00000000") -> None:
    payload = {
        "nom": nom,
        "prenom": prenom,
        "email": email,
        "password": password,
        "telephone": telephone,
    }
    r = requests.post(f"{API_URL}/auth/register", json=payload)
    if r.status_code != 200:
        raise AssertionError(f"Register failed: {r.status_code} {r.text}")
    return None

def api_login_session(email: str, password: str) -> requests.Session:
    s = requests.Session()
    r = s.post(f"{API_URL}/auth/login", json={"email": email, "password": password})
    r.raise_for_status()
    token = r.json()["token"]
    s.headers.update({"Authorization": f"Bearer {token}"})
    return s

def api_logout(s: requests.Session) -> None:
    r = s.post(f"{API_URL}/auth/logout")
    if r.status_code not in (200, 204):
        raise AssertionError(f"Logout failed: {r.status_code} {r.text}")

In [ ]:
# TESTS API (simples)
print("--- API: INSCRIPTION ---")
api_register(NEW_EMAIL, NEW_PASSWORD, nom="Patient", prenom="Auto", telephone="11111111")
print("INSCRIPTION API OK")

print("--- API: LOGIN (nouveau compte) ---")
s_new = api_login_session(NEW_EMAIL, NEW_PASSWORD)
print("LOGIN API (nouveau compte) OK")
api_logout(s_new)
print("LOGOUT API (nouveau compte) OK")

print("--- API: TESTS ADMIN (exemples métier) ---")
s = api_login_session(EMAIL, PASSWORD)
print("LOGIN API (admin) OK")

medecin_payload = {
    "nom": "Ben Ali",
    "prenom": "Ahmed",
    "specialite": "Cardiologue",
    "telephone": "22123456"
}
r = s.post(f"{API_URL}/medecins", json=medecin_payload)
assert r.status_code in (200, 201), (r.status_code, r.text)
print("AJOUT MEDECIN API OK")

r = s.get(f"{API_URL}/medecins")
r.raise_for_status()
medecins = r.json()
print("LISTE MEDECINS API OK — count =", len(medecins) if isinstance(medecins, list) else "?")

ordonnance_payload = {
    "patientId": 1,
    "medecinId": 1,
    "description": "Paracétamol 1g - 3 fois par jour",
    "date": "2026-01-18"
}
r = s.post(f"{API_URL}/ordonnances", json=ordonnance_payload)
assert r.status_code in (200, 201), (r.status_code, r.text)
print("AJOUT ORDONNANCE API OK")

api_logout(s)
print("LOGOUT API (admin) OK")

In [ ]:
def start_driver() -> tuple[webdriver.Chrome, WebDriverWait]:
    options = Options()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    wait = WebDriverWait(driver, 10)
    return driver, wait

In [ ]:
def click_any(driver, wait, selectors: list[str], desc: str = "element") -> None:
    last_err = None
    for css in selectors:
        try:
            el = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, css)))
            el.click()
            return
        except Exception as e:
            last_err = e
    raise AssertionError(f"Impossible de cliquer {desc}. Selectors testés: {selectors}. Last error: {last_err}")

def click_button_by_text(driver, wait, texts: list[str]) -> None:
    # Support <button> ou <a> (lien stylé comme bouton)
    for t in texts:
        xpath = f"//button[contains(normalize-space(.), '{t}')] | //a[contains(normalize-space(.), '{t}')]"
        try:
            el = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
            el.click()
            return
        except Exception:
            pass
    raise AssertionError(f"Aucun bouton/lien trouvé avec les textes: {texts}")

def click_submit(driver, wait) -> None:
    click_any(
        driver,
        wait,
        selectors=[
            "button[type='submit']",
            "button.btn-submit",
            "button.btn-primary",
            "button.btn.btn-primary",
            "button.btn-success",
            "form button",
            "input[type='submit']",
            "a.btn.btn-primary",
        ],
        desc="submit",
    )

def ui_go_to_register_from_login(driver, wait) -> None:
    driver.get(f"{FRONT_URL}/login")
    click_button_by_text(driver, wait, ["Créer un compte", "Inscription", "Register"])
    wait.until(EC.url_contains("/register"))

def ui_go_to_login_from_register(driver, wait) -> None:
    driver.get(f"{FRONT_URL}/register")
    click_button_by_text(driver, wait, ["Se connecter", "Connexion", "Login"])
    wait.until(EC.url_contains("/login"))

def ui_go_home_from_register(driver, wait) -> None:
    driver.get(f"{FRONT_URL}/register")
    click_button_by_text(driver, wait, ["Retour à l'accueil", "Accueil", "Home"])
    wait.until(EC.url_contains("/home"))

def ui_register(driver, wait, nom: str, prenom: str, email: str, telephone: str, password: str) -> None:
    driver.get(f"{FRONT_URL}/register")

    def fill_by_css(selector: str, value: str) -> None:
        el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
        el.clear()
        el.send_keys(value)

    fill_by_css("input.form-input[placeholder='Martin']", nom)
    fill_by_css("input.form-input[placeholder='Sophie']", prenom)
    fill_by_css("input.form-input[placeholder='sophie.martin@exemple.com']", email)
    fill_by_css("input.form-input[placeholder^='+']", telephone)
    fill_by_css("input.form-input[type='password']", password)

    try:
        click_submit(driver, wait)
    except AssertionError:
        click_button_by_text(driver, wait, ["Créer un compte", "S'inscrire", "Inscription"])

    wait.until(EC.url_contains("/login"))

def ui_login(driver, wait, email: str, password: str) -> None:
    driver.get(f"{FRONT_URL}/login")

    email_el = wait.until(EC.presence_of_element_located((By.NAME, "email")))
    email_el.clear()
    email_el.send_keys(email)

    pass_el = wait.until(EC.presence_of_element_located((By.NAME, "password")))
    pass_el.clear()
    pass_el.send_keys(password)

    try:
        click_submit(driver, wait)
    except AssertionError:
        click_button_by_text(driver, wait, ["Se connecter", "Connexion", "Login"])

    wait.until(EC.url_contains("dashboard"))

def ui_add_medecin(driver, wait) -> None:
    driver.get(f"{FRONT_URL}/medecins/add")
    wait.until(EC.presence_of_element_located((By.NAME, "nom"))).send_keys("Salah")
    driver.find_element(By.NAME, "prenom").send_keys("Mohamed")
    driver.find_element(By.NAME, "specialite").send_keys("Dentiste")
    driver.find_element(By.NAME, "telephone").send_keys("98765432")

    current_url = driver.current_url
    try:
        click_submit(driver, wait)
    except AssertionError:
        click_button_by_text(driver, wait, ["Ajouter", "Enregistrer", "Valider"])

    try:
        wait.until(lambda d: d.current_url != current_url)
    except TimeoutException:
        pass

def ui_add_ordonnance(driver, wait) -> None:
    driver.get(f"{FRONT_URL}/ordonnances/add")
    wait.until(EC.presence_of_element_located((By.NAME, "description"))).send_keys("Ibuprofène 400mg")

    current_url = driver.current_url
    try:
        click_submit(driver, wait)
    except AssertionError:
        click_button_by_text(driver, wait, ["Ajouter", "Enregistrer", "Valider"])

    try:
        wait.until(lambda d: d.current_url != current_url)
    except TimeoutException:
        pass

def ui_logout(driver, wait) -> None:
    driver.find_element(By.ID, "logout").click()
    wait.until(EC.url_contains("login"))

In [ ]:
# TESTS UI (simples)
driver, wait = start_driver()
try:
    print("--- UI: 3 BOUTONS/Liens ---")
    ui_go_to_register_from_login(driver, wait)
    print("BOUTON: Login -> Register OK")
    ui_go_to_login_from_register(driver, wait)
    print("BOUTON: Register -> Login OK")
    ui_go_home_from_register(driver, wait)
    print("BOUTON: Register -> Home OK")

    print("--- UI: INSCRIPTION ---")
    ui_register(driver, wait, nom="Patient", prenom="Auto", email=NEW_EMAIL, telephone="+33 6 00 00 00 00", password=NEW_PASSWORD)
    print("INSCRIPTION UI OK")

    print("--- UI: LOGIN (nouveau compte) ---")
    ui_login(driver, wait, NEW_EMAIL, NEW_PASSWORD)
    print("LOGIN UI (nouveau compte) OK")
    ui_logout(driver, wait)
    print("LOGOUT UI (nouveau compte) OK")

    print("--- UI: SCENARIO ADMIN (exemples métier) ---")
    ui_login(driver, wait, EMAIL, PASSWORD)
    print("LOGIN UI (admin) OK")

    ui_add_medecin(driver, wait)
    print("AJOUT MEDECIN UI OK")

    ui_add_ordonnance(driver, wait)
    print("AJOUT ORDONNANCE UI OK")

    ui_logout(driver, wait)
    print("LOGOUT UI (admin) OK")
finally:
    driver.quit()

print("TESTS TERMINÉS")